# Milestone 3: Cleaning/Formating Website data

Author: Rosie Navarro

Date: October 30, 2026


In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import requests 
from bs4 import BeautifulSoup
from urllib.parse import urljoin

## Step 1: 

The information from the website can be seen, but it is not formated in as a table. To format it to a table, a loop can be made to search for product information for each entry. An issue that I was having with the website was that the data was only load for the first page. To fix the issue, a for loop was created to look at all the information from all the pages. 


In [57]:
products = []

for page in range(1, 22):

    response = requests.get(
        "https://scrapingsandbox.com/",
        params={"page": page}
    )

    soup = BeautifulSoup(response.text, "html.parser")

    for card in soup.find_all("a", class_="product-card"):

        original_price = card.find(class_="compare-at-price")

        product = {
            "Vendor": card.find(class_="vendor").get_text(strip=True),
            "Rating": card.find(class_="rating").get_text(strip=True),
            "Product": card.find(class_="product-name").get_text(strip=True),
            "Price": card.find(class_="price").get_text(" ", strip=True),
            "Original Price": original_price.get_text(" ", strip=True) if original_price else None,
            "Category": card.find(class_="category").get_text(strip=True),
            "SKU": card.find(class_="sku").get_text(strip=True),
            "Availability": card.find(class_="availability").get_text(strip=True),
            "URL": urljoin(response.url, card["href"])
        }

        products.append(product)

website_df = pd.DataFrame(products)

website_df.head(50)

,Vendor,Rating,Product,Price,Original Price,Category,SKU,Availability,URL
0,SoundWave,4,Lightweight Probiotics,$ 155.62,$ 206.69,Health,SKU-HEA-0001,In Stock,https://scrapingsandbox.com/product/1
1,SoundWave,4.6,Wireless LED Desk Lamp,$ 145.40,$ 210.01,Electronics,SKU-ELE-0002,OUT OF STOCK,https://scrapingsandbox.com/product/2
2,StyleCraft,3.5,Deluxe Tea Collection,$ 185.01,$ 313.98,Food & Beverage,SKU-FOO-0003,In Stock,https://scrapingsandbox.com/product/3
3,TechVault,3.6,Essential Tire Gauge,$ 70.86,NaN,Automotive,SKU-AUT-0004,OUT OF STOCK,https://scrapingsandbox.com/product/4
4,SparkleBox,3.5,Smart Carving Tools,$ 97.13,NaN,Art,SKU-ART-0005,In Stock,https://scrapingsandbox.com/product/5
5,StyleCraft,4.6,Professional Dog Food,$ 145.40,NaN,Pet Supplies,SKU-PET-0006,In Stock,https://scrapingsandbox.com/product/6
6,UrbanThread,3,Smart Car Wash Kit,$ 139.94,$ 173.26,Automotive,SKU-AUT-0007,In Stock,https://scrapingsandbox.com/product/7
7,AutoParts Pro,4.5,Premium Bluetooth Speaker,$ 108.99,NaN,Electronics,SKU-ELE-0008,In Stock,https://scrapingsandbox.com/product/8
8,GlowUp,3.4,Vintage Tablet Stand,$ 44.15,$ 69.14,Electronics,SKU-ELE-0009,In Stock,https://scrapingsandbox.com/product/9
9,VitalLife,4.2,Portable Marker Pack,$ 193.91,$ 258.48,Art,SKU-ART-0010,In Stock,https://scrapingsandbox.com/product/10


head(50) was use to make sure that the data is being loaded correctly. 

## Step 2: Data Types
Using info we can see the data types for all of the variables. One interesting thing about this dataset is that all of them are strings. Some of the variables need to be converted to numeric data types, such as Rating, Price, and Original Price. 

In [58]:
website_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 504 entries, 0 to 503
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Vendor          504 non-null    str  
 1   Rating          504 non-null    str  
 2   Product         504 non-null    str  
 3   Price           504 non-null    str  
 4   Original Price  210 non-null    str  
 5   Category        504 non-null    str  
 6   SKU             504 non-null    str  
 7   Availability    504 non-null    str  
 8   URL             504 non-null    str  
dtypes: str(9)
memory usage: 35.6 KB


In [59]:
website_df["Price"] = (
    website_df["Price"].str.replace("$", "", regex=False).str.replace(",", "", regex=False).astype(float))

website_df["Original Price"] = (
    website_df["Original Price"].str.replace("$", "", regex=False).str.replace(",", "", regex=False).astype(float))

In [60]:
website_df["Rating"] = website_df["Rating"].astype(float)

To make sure the change was made, info is used again to see the data type for the variables. 

In [61]:
website_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 504 entries, 0 to 503
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Vendor          504 non-null    str    
 1   Rating          504 non-null    float64
 2   Product         504 non-null    str    
 3   Price           504 non-null    float64
 4   Original Price  210 non-null    float64
 5   Category        504 non-null    str    
 6   SKU             504 non-null    str    
 7   Availability    504 non-null    str    
 8   URL             504 non-null    str    
dtypes: float64(3), str(6)
memory usage: 35.6 KB


## Step 3: Null Values

The next step is to find out if there are any missing values in the dataset and deciding what should be done to those values. 

info can be used to see all of the variables in the dataset and see how many non-null values they contain. The dataset contains a total of 504 entries. Out of all of the variables original prices is the only variable that contains null values. Info also gives us information on the data type of all of the variables. 

isnull() is used to verify the information above. 

In [62]:
website_df.isnull().sum()

Vendor              0
Rating              0
Product             0
Price               0
Original Price    294
Category            0
SKU                 0
Availability        0
URL                 0
dtype: int64

By looking at the data, we can see that price and original price can be used to see if the item is on sale or not. Null original price values will be changed to the value that price is, indicating that there is not a sale for that specific item. 

To confirm that the change was made isnull sum will be used again, it should be 0. 

In [63]:
website_df["Original Price"] = website_df["Original Price"].fillna(website_df["Price"])
# Null original price to price

website_df["Original Price"].isnull().sum() #null should be 0

np.int64(0)

## Step 4: Duplicates

The next step is to check if there are any duplicates in the dataset, in other words, the same product. 

In [64]:
website_df.duplicated().sum()

np.int64(480)

In [65]:
website_df["SKU"].nunique()

24

This is where an important issue comes up. Using info we know that there is a total of 504 entries, but from those entries there are only 24 unique values. With that in mind that means that the same 24 entries are repeated over and over until it reaches the 504 entries, this is what is causing the duplicates. 

For this assignment the the duplicates will be deleted (the additional data will be used later on but not for this assignment). 

In [66]:
website_df = website_df.drop_duplicates(subset="SKU", keep="first")

In [69]:
website_df.head(24)

,Vendor,Rating,Product,Price,Original Price,Category,SKU,Availability,URL
0,SoundWave,4.0,Lightweight Probiotics,155.62,206.69,Health,SKU-HEA-0001,In Stock,https://scrapingsandbox.com/product/1
1,SoundWave,4.6,Wireless LED Desk Lamp,145.40,210.01,Electronics,SKU-ELE-0002,OUT OF STOCK,https://scrapingsandbox.com/product/2
2,StyleCraft,3.5,Deluxe Tea Collection,185.01,313.98,Food & Beverage,SKU-FOO-0003,In Stock,https://scrapingsandbox.com/product/3
3,TechVault,3.6,Essential Tire Gauge,70.86,70.86,Automotive,SKU-AUT-0004,OUT OF STOCK,https://scrapingsandbox.com/product/4
4,SparkleBox,3.5,Smart Carving Tools,97.13,97.13,Art,SKU-ART-0005,In Stock,https://scrapingsandbox.com/product/5
5,StyleCraft,4.6,Professional Dog Food,145.40,145.40,Pet Supplies,SKU-PET-0006,In Stock,https://scrapingsandbox.com/product/6
6,UrbanThread,3.0,Smart Car Wash Kit,139.94,173.26,Automotive,SKU-AUT-0007,In Stock,https://scrapingsandbox.com/product/7
7,AutoParts Pro,4.5,Premium Bluetooth Speaker,108.99,108.99,Electronics,SKU-ELE-0008,In Stock,https://scrapingsandbox.com/product/8
8,GlowUp,3.4,Vintage Tablet Stand,44.15,69.14,Electronics,SKU-ELE-0009,In Stock,https://scrapingsandbox.com/product/9
9,VitalLife,4.2,Portable Marker Pack,193.91,258.48,Art,SKU-ART-0010,In Stock,https://scrapingsandbox.com/product/10


## Step 5: Replacing Headers

Since the data will be used with other datasets, the varible names will be changed so that they match with the other variables. 


In [73]:
website_df = website_df.rename(columns={
    "product": "product",
    "original_price": "originalprice",
    "vendor": "brand",
    "sku": "sku",
    "availability": "availability",
    "url": "url"})

In [74]:
website_df.head()

,Vendor,Rating,Product,Price,Original Price,Category,SKU,Availability,URL
0,SoundWave,4.0,Lightweight Probiotics,155.62,206.69,Health,SKU-HEA-0001,In Stock,https://scrapingsandbox.com/product/1
1,SoundWave,4.6,Wireless LED Desk Lamp,145.40,210.01,Electronics,SKU-ELE-0002,OUT OF STOCK,https://scrapingsandbox.com/product/2
2,StyleCraft,3.5,Deluxe Tea Collection,185.01,313.98,Food & Beverage,SKU-FOO-0003,In Stock,https://scrapingsandbox.com/product/3
3,TechVault,3.6,Essential Tire Gauge,70.86,70.86,Automotive,SKU-AUT-0004,OUT OF STOCK,https://scrapingsandbox.com/product/4
4,SparkleBox,3.5,Smart Carving Tools,97.13,97.13,Art,SKU-ART-0005,In Stock,https://scrapingsandbox.com/product/5


## Ethical Implications: 

The data was obtained from scraping sandbox, which had variables like vendor, rating, product name, price, category, SKU, availability, and url. There were five things that were done to this website, making of the table, converting data types, fixing null values, removing duplicates, and charging variable names. 

SKU was used to identify duplicates and the duplicates were removed for this assignment. This is also where the main issue occured. The same 24 products were collected multiple times until it reached 304 entries. Later on I will try to figure out how to include all of the data without having any duplicates. Since the data was created for scraping, there are not ethical implications from the data collectiong, but it this is done with other retail stores, this could have ethical implications. Ince this is practice data, there is no customer information. Just to make sure that the data was created ethically, i will verify the information from the website. 
